In [ ]:
import os
import csv
import numpy as np
import xarray as xr
import scipy.io as sio
from scipy.interpolate import interp1d

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.ticker import MultipleLocator, FixedLocator
from matplotlib.colors import to_hex, to_rgb
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# All data this notebook reads is repo-relative: data/processed/ for the MATLAB pipeline's
# output and data/external/ for the LR04 stack. So there is no PROXY_DATA_DIR here -- that
# variable pointed at the raw proxy tree, which only the MATLAB stage reads. Launch Jupyter
# from the repo root and the paths below resolve with no setup at all.
extern = '../data/external'
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR')
if not opath:
    raise RuntimeError(
        "FIG_OUTPUT_DIR is not set. Run `source config/paths.env` before launching Jupyter "
        "(see config/paths.env.example). Refusing to guess a default: the old fallback wrote to "
        "a repo-local outputs/ that tools/sync_manuscript_figs.sh never read, and figures "
        "silently diverged between the two directories."
    )
os.makedirs(opath, exist_ok=True)

In [ ]:
# --- LOAD PROXY DATA --- #

# Read straight from the MATLAB pipeline's output.
# The scripts/matlab/dDwax_data_processing_{nh22p,d480_d479}.m write the *_FAMEs_current.mat files
# that carry every field this notebook needs: age, dDraw, stdev, dDivc, dDp, jas
#
# *_FAMEs_current.mat is deliberately undated: the MATLAB saves both a dated archive copy and
# this stable name, so nothing here has to name a date and quietly go stale

def load_fames(path, key):
    """Load a *_FAMEs_current.mat struct as the xr.Dataset the rest of this notebook expects.

    MATLAB structs come out of loadmat as nested object arrays; simplify_cells=True unwraps
    them to a plain dict of arrays and squeezes the (n,1) columns to (n,). The dDp and jas
    ensembles keep their own ensemble dimensions because they differ in width -- dDp is 1000
    Monte Carlo draws, jas is 4000 (10 Gibbs chains x 800, thinned by 2).
    """
    s = sio.loadmat(path, simplify_cells=True)[key]
    ds = xr.Dataset(
        {v: ('age', s[v]) for v in ('dDraw', 'stdev', 'dDivc')},
        coords={'age': s['age']},
    )
    # 'jas' is the MATLAB field name; 'pJAS' is what this notebook calls it
    for name, mat in (('dDp', s['dDp']), ('pJAS', s['jas'])):
        dim = f'ensemble_n_{name}'
        ds = ds.assign_coords({dim: np.arange(mat.shape[1])})
        ds[name] = (('age', dim), mat)
    return ds

procd = '../data/processed'

# nh22p
nh22p = load_fames(f'{procd}/nh22p_FAMEs_current.mat', 'nh22p')

# dsdp-480-479 
# the saved struct is already the 147-sample d480+d479 composite, spliced and
# age-sorted in MATLAB (dDwax_data_processing_d480_d479.m), not assembled here
d480_479 = load_fames(f'{procd}/Guaymas_d480_d479_FAMEs_current.mat', 'Guay')

# lr04
# Read from data/external/lr04.mat: `delob` columns are [age (ka), d18O (per mil), error]
delob = sio.loadmat(f'{extern}/lr04.mat')['delob']
lr04_age = delob[:, 0]
lr04_d18O = delob[:, 1]

# Interpolated onto a proxy record's own age axis wherever LR04 is needed at proxy resolution
# (the correlation cells below). interp1d(...,'linear','extrap') rather than np.interp (which
# clamps flat outside its domain) matches MATLAB's interp1(...,'linear','extrap') in
# SST_dD_correlation.m -- likely a no-op in practice since every proxy age falls well inside
# LR04's 0-5320 ka span, but kept explicit. Defined once, here, rather than inside either
# per-core correlation cell below, so it doesn't matter which of those two cells runs first.
lr04_interp = interp1d(lr04_age, lr04_d18O, kind='linear', fill_value='extrapolate')

In [ ]:
# --- INTERVAL DIFFS FOR dDraw TIMESERIES --- #

cores     = ['d480_479','nh22p']
intervals = ['late_holocene','holocene','lgm','lig','pgm']

# init dictionaries
age_bnds        = { interval: {} for interval in intervals }
dDtimeslice     = { core: { interval: {} for interval in intervals } for core in cores }
interval_mean   = { core: { interval: {} for interval in intervals } for core in cores }
interval_stddev = { core: { interval: {} for interval in intervals } for core in cores }
differences     = { core: { modern_i: { i: {} for i in ['lgm','lig','pgm'] } for modern_i in ['late_holocene','holocene'] } for core in cores }

# populate age bounds (ka)
age_bnds['late_holocene'] = [0, 4]
age_bnds['holocene']      = [0, 11.7]
age_bnds['lgm']           = [18, 24]
age_bnds['lig']           = [117, 130]
age_bnds['pgm']           = [135, 150]

for core in cores:
    print(f'\n{core}')
    for interval in intervals:
        # subset dD wax time-series
        t_min, t_max = age_bnds[interval][0], age_bnds[interval][1]
        if core=='d480_479':
            timeslice_subset = d480_479.sel(age=slice(t_min,t_max))
        elif core=='nh22p':
            timeslice_subset = nh22p.sel(age=slice(t_min,t_max))
        dDtimeslice[core][interval] = timeslice_subset
        # calculate timeslice stats
        interval_mean[core][interval]=dDtimeslice[core][interval].dDraw.mean(dim=['age'])
        interval_stddev[core][interval]=dDtimeslice[core][interval].dDraw.std(dim=['age'])
        print(f'{interval} mean, std dev = {np.round(interval_mean[core][interval],2).values}, {np.round(interval_stddev[core][interval],2).values}')

# print interval differences
for core in cores:
    print(f'\n{core}')
    for modern_ref in ['late_holocene','holocene']:
        for interval in ['lgm','lig','pgm']:
            differences[core][modern_ref][interval] = np.round(interval_mean[core][interval].values - interval_mean[core][modern_ref].values,2)
            print(f'{interval} - {modern_ref} = {differences[core][modern_ref][interval]}')

In [ ]:
### create data frame for exporting
# Writes to data/processed/
# Path is relative to the repo root, which is where notebooks must be launched from.
# These CSVs are tracked in git — they are how the Casper clone receives proxy data.
#
# Header and rows are built from `cores` and `intervals`
#
# Column suffix is `_stddev`:
# the quantity is the standard deviation (i.e. the spread of the data) 
# of the dDraw samples falling inside each age window (`.std(dim=['age'])` above)

core_meta = {                        # dict key -> (csv core_name, lon, lat)
    'd480_479': ('DSDP_480_479', -111.62,   27.85),
    'nh22p':    ('NH22P',        -106.5183, 22.5183),
}

header = ['core_name', 'lon', 'lat']
for interval in intervals:
    header += [f'{interval}_dD', f'{interval}_dD_stddev']

data = [header]
for core in cores:
    core_name, lon, lat = core_meta[core]
    row = [core_name, f'{lon}', f'{lat}']
    for interval in intervals:
        row += [f"{float(interval_mean[core][interval].values)}",
                f"{float(interval_stddev[core][interval].values)}"]
    data.append(row)

with open('../data/processed/timeslice_mean_proxy_dDraw.csv', "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(data)


In [ ]:
# --- INTERVAL DIFFS FOR dDp TIMESERIES --- #

cores     = ['d480_479','nh22p']
intervals = ['late_holocene','holocene','lgm','lig','pgm']

# init dictionaries
age_bnds        = { interval: {} for interval in intervals }
dDtimeslice     = { core: { interval: {} for interval in intervals } for core in cores }
interval_mean   = { core: { interval: {} for interval in intervals } for core in cores }
interval_stddev = { core: { interval: {} for interval in intervals } for core in cores }
differences     = { core: { modern_i: { i: {} for i in ['lgm','lig','pgm'] } for modern_i in ['late_holocene','holocene'] } for core in cores }

# populate age bounds (ka)
age_bnds['late_holocene'] = [0, 4]
age_bnds['holocene']      = [0, 11.7]
age_bnds['lgm']           = [18, 24]
age_bnds['lig']           = [117, 130]
age_bnds['pgm']           = [135, 150]

for core in cores:
    print(f'\n{core}')
    for interval in intervals:
        # subset dD wax time-series
        t_min, t_max = age_bnds[interval][0], age_bnds[interval][1]
        if core=='d480_479':
            timeslice_subset = d480_479.sel(age=slice(t_min,t_max))
        elif core=='nh22p':
            timeslice_subset = nh22p.sel(age=slice(t_min,t_max))
        dDtimeslice[core][interval] = timeslice_subset
        # calculate timeslice stats
        interval_mean[core][interval]=dDtimeslice[core][interval].dDp.median(dim='ensemble_n_dDp').mean(dim=['age'])
        interval_stddev[core][interval]=dDtimeslice[core][interval].dDp.median(dim='ensemble_n_dDp').std(dim=['age'])
        print(f'{interval} mean, std dev = {np.round(interval_mean[core][interval],1).values}, {np.round(interval_stddev[core][interval],1).values}')

# print interval differences
for core in cores:
    print(f'\n{core}')
    for modern_ref in ['late_holocene','holocene']:
        for interval in ['lgm','lig','pgm']:
            differences[core][modern_ref][interval] = np.round(interval_mean[core][interval].values - interval_mean[core][modern_ref].values,1)
            print(f'{interval} - {modern_ref} = {differences[core][modern_ref][interval]}')

In [ ]:
### create data frame for exporting
# Writes to data/processed/
# Path is relative to the repo root, which is where notebooks must be launched from.
# These CSVs are tracked in git — they are how the Casper clone receives proxy data.
#
# Header and rows are built from `cores` and `intervals`
#
# Column suffix is `_stddev`:
# the quantity is the standard deviation (i.e. the spread of the data) 
# of the dDraw samples falling inside each age window (`.std(dim=['age'])` above)

core_meta = {                        # dict key -> (csv core_name, lon, lat)
    'd480_479': ('DSDP_480_479', -111.62,   27.85),
    'nh22p':    ('NH22P',        -106.5183, 22.5183),
}

header = ['core_name', 'lon', 'lat']
for interval in intervals:
    header += [f'{interval}_dD', f'{interval}_dD_stddev']

data = [header]
for core in cores:
    core_name, lon, lat = core_meta[core]
    row = [core_name, f'{lon}', f'{lat}']
    for interval in intervals:
        row += [f"{float(interval_mean[core][interval].values)}",
                f"{float(interval_stddev[core][interval].values)}"]
    data.append(row)

with open('../data/processed/timeslice_mean_proxy_dDp.csv', "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(data)


## Plot Timeseries

In [ ]:
# --- Shared figure plumbing, used by EVERY timeseries figure below ---------------------------
#
# Both figures used to carry their own copy of the MIS-ribbon drawing, and both copies had the
# same two defects:
#
#   1. The ribbon was drawn INSIDE a data panel, as rectangles positioned in delta-D units above
#      ymax and escaping the panel via clip_on=False. Its vertical position was therefore
#      coupled to the data limits, and it landed on top of the topmost y tick label.
#   2. Tick sets that included the panel's own y limits. With hspace=0 the top panel's lowest
#      tick label and the lower panel's highest land on the same pixel row and overprint. In the
#      dDraw figure, -163 and -133 rendered as an unreadable "-183".
#
# Fixed once, here, so the two figures cannot drift apart again: mis_ribbon() puts the ribbon in
# its OWN axes with ylim (0,1), and every figure below uses interior ticks that never touch a
# shared panel edge.

COLD_MIS        = np.array([[14,29],[38,45],[57,71],[84,95],[105,114],[135,141],[141,150]])
COLD_MIS_LABELS = ['2', '3b', '4', '5b', '5d', '6a', '6b']
WARM_MIS        = np.array([[0,14],[29,38],[45,57],[71,84],[95,105],[114,135]])
WARM_MIS_LABELS = ['1', '3a', '3c', '5a', '5c', '5e']
INTGLCL         = np.array([[0,11.7],[117,130]])
INTGLCL_LABELS  = ['HOL', 'LIG']

# age axis, shared by every panel
XMIN, XMAX = 0, 150

# --- palette ---------------------------------------------------------------------------------
# Line colours are sampled from cmocean `rain`, whose blue-green stretch runs ~0.42-0.78 (below
# that it is tan, above it navy/violet). 0.42 and 0.72 were picked by sweeping every pair >=0.12
# apart through a CVD/contrast validator and taking the best scorer: separation dE 20.4 (protan)
# / 18.9 (tritan) and 21.1 normal-vision, so the two cores stay distinguishable to colour-blind
# readers and in greyscale. Don't re-pick these by eye.
TEAL  = to_hex(cmo.rain(0.72))                            # DSDP-480/479
GREEN = to_hex(cmo.rain(0.42))                            # NH22P
BAND, BAND_ALPHA, BAND_LABEL = '#f7efd7', 1, '#8a6a15'  # interglacial highlight
WARM_FC, COLD_FC = '#fbfbf9', '#dcdcd6'                    # MIS ribbon fills
LR04_C = '0.62'                                            # benthic stack: recessive by design


def tint(color, amount):
    """Mix `color` toward white by `amount` (0 = unchanged, 1 = white).

    Used for the error envelopes, so envelope and line read as one series rather than two.
    """
    r, g, b = to_rgb(color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)


def mis_ribbon(ax, xmin=XMIN, xmax=XMAX):
    """Draw the MIS stratigraphy bar into its OWN axes (data coords: age by 0..1).

    Give this a dedicated gridspec row. Do NOT draw it inside a data panel -- that is defect 1
    above.

    If the host panels use an x range wider than (xmin, xmax) -- a right-hand label margin, say
    -- pass that same range here. A ribbon on (0,150) beside panels on (0,196) silently loses
    registration: the stage boundaries stop lining up with the ages beneath them.
    """
    for spans, labels, fc in ((WARM_MIS, WARM_MIS_LABELS, WARM_FC),
                              (COLD_MIS, COLD_MIS_LABELS, COLD_FC)):
        for (x0, x1), lab in zip(spans, labels):
            ax.add_patch(plt.Rectangle((x0, 0), x1 - x0, 1, fc=fc, ec='0.35', lw=0.5, zorder=2))
            ax.text((x0 + x1) / 2, 0.5, lab, ha='center', va='center', size=8.5,
                    weight='bold', color='0.15', zorder=3)
    ax.set(xlim=(xmin, xmax), ylim=(0, 1), xticks=[], yticks=[])
    for s in ax.spines.values():
        s.set_visible(False)


def interglacial_bands(ax, labels=False, label_y=0.045):
    """Shade the HOL and LIG windows on ax, optionally labelling them.

    Labels use a blended transform (x in ka, y in axes fraction) so they sit clear of a panel
    junction whatever the panel's y units are.
    """
    for (x0, x1) in INTGLCL:
        ax.axvspan(x0, x1, color=BAND, alpha=BAND_ALPHA, lw=0, zorder=0)
    if labels:
        blend = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
        for (x0, x1), lab in zip(INTGLCL, INTGLCL_LABELS):
            ax.text((x0 + x1) / 2, label_y, lab, transform=blend, ha='center', va='bottom',
                    size=10, weight='normal', color=BAND_LABEL, zorder=0)


def slot_ylim(lo, hi, slot_lo, slot_hi, invert=False):
    """ylim placing data range [lo,hi] into the vertical slot [slot_lo,slot_hi] of the axes.

    This is what makes the stacked version's traces stack and overlap: each trace keeps its own
    units, but its data occupies only a chosen band of the shared plot area.

    invert=True is for a downward-running axis (LR04 d18O, cold at the bottom). Flipping the
    returned pair is NOT sufficient -- that also mirrors the slot to 1-slot, which puts the
    trace at the wrong end of the figure. The bottom edge has to be solved for directly.
    """
    span = (hi - lo) / (slot_hi - slot_lo)
    if not invert:
        y0 = lo - slot_lo * span                # `lo` sits at slot_lo, `hi` at slot_hi
        return (y0, y0 + span)
    y_bottom = hi + slot_lo * span              # `hi` sits at slot_lo, `lo` at slot_hi
    return (y_bottom, y_bottom - span)

## dDraw timeseries

In [ ]:
# --- dDraw axis limits, shared by both versions of the figure below --------------------------

# Interior ticks only -- they must never sit on a panel's own y limit, or the top panel's lowest
# label and the lower panel's highest overprint at the hspace=0 junction. See the shared cell.
DD_LIM = (-164, -133)
DD_TICKS    = [-160, -155, -150, -145, -140, -135]
DD_TICKS_NH = [-160, -155, -150, -145, -140, -135]   # NH22P's own bracket in the stacked version, offset from DSDP's

LR04_LIM   = (5.15, 2.95)
LR04_TICKS = [5.0, 4.5, 4.0, 3.5, 3.0]       # interior: 5.0 and 3.0 would sit on the panel limits

In [ ]:
# === VERSION A: two core panels, LR04 twinned under each =====================================
# The original structure, with the MIS ribbon moved into its own gridspec row and interior
# ticks on both the left and the right axes.

fig = plt.figure(figsize=(9, 6))
gs = fig.add_gridspec(3, height_ratios=[0.5, 5, 5], hspace=0)

# The ribbon deliberately does NOT sharex with the data panels. Sharing ties its tick locator to
# theirs, so setting the 20-ka major locator on the bottom panel pushes tick labels back onto
# the ribbon. It takes the same xlim instead.
ribbon = fig.add_subplot(gs[0])
mis_ribbon(ribbon)

axes = []
for i, (name, core, col) in enumerate([('DSDP-480/479', d480_479, TEAL),
                                       ('NH22P', nh22p, GREEN)]):
    ax = fig.add_subplot(gs[i + 1], sharex=axes[0] if axes else None)
    axes.append(ax)

    #interglacial_bands(ax)

    # benthic stack, behind the proxy on a twin axis
    axr = ax.twinx()
    axr.plot(lr04_age, lr04_d18O, color=LR04_C, lw=1.1, zorder=100)
    axr.set(xlim=(XMIN, XMAX), ylim=LR04_LIM, yticks=LR04_TICKS)
    axr.set_ylabel(u'LR04 $\\delta^{18}$O$_b$ [‰]', rotation=270, labelpad=16, size=10,
                   color=LR04_C)
    axr.tick_params(axis='y', colors=LR04_C, labelsize=9, direction='out')
    axr.spines['right'].set_color(LR04_C)
    axr.spines[['top', 'left', 'bottom']].set_visible(False)
    ax.set_zorder(axr.get_zorder() + 1)   # proxy in front of the stack
    ax.patch.set_visible(False)

    # proxy series: envelope is the line's own hue mixed toward white
    ax.fill_between(core.age, core.dDraw - core.stdev, core.dDraw + core.stdev,
                    color=tint(col, 0.55), lw=0, alpha=0.9, zorder=101)
    ax.plot(core.age, core.dDraw, color=col, lw=1.5, zorder=102, solid_capstyle='round')

    ax.set(xlim=(XMIN, XMAX), ylim=DD_LIM, yticks=DD_TICKS)
    ax.set_ylabel(u'$\\delta$D$_{C30}$ [‰]', size=10.5, labelpad=4)
    ax.tick_params(axis='both', direction='out', labelsize=9.5)
    ax.yaxis.set_minor_locator(MultipleLocator(5))
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_position(('outward', 2))
    # panel name in the core's own colour, so identity is never colour-alone
    ax.text(0.012, 0.93, name, transform=ax.transAxes, ha='left', va='top', size=12,
            weight='bold', color=col, zorder=7)

axes[0].tick_params(labelbottom=False)     # x labels on the bottom panel only
bottom = axes[-1]
bottom.set_xlabel('AGE [ka]', size=10.5, labelpad=4)
bottom.xaxis.set_major_locator(MultipleLocator(20))
ribbon.set_xlim(XMIN, XMAX)

#interglacial_bands(bottom, labels=True)    # HOL / LIG named once, at the foot of the figure

fig.savefig(f'{opath}/dDraw_timeseries.pdf', bbox_inches='tight')

In [ ]:
# === VERSION B: stacked overlapping traces, one plot area ====================================
# LR04 first (top), then the two cores. Every trace keeps its own units and its own
# colour-matched y bracket; slot_ylim() maps each into a vertical band of the shared axes.

# Slots, top to bottom, as (bottom, top) fractions of the plot area. They overlap on purpose --
# that overlap is what clusters the traces. All three are the SAME height: a shorter LR04 slot
# was tried and reverted -- it drew LR04's axis bracket over the same data range but a smaller
# slot, so a given span of the bracket represented less vertical space than the equivalent span
# on the core brackets, making LR04 read as flatter than it actually is relative to the data.
SLOTS = [(0.56, 1.00), (0.28, 0.72), (0.00, 0.44)]

# Trace labels sit INSIDE the plot area, in each trace's own colour, in the whitespace beside
# the curve they name: (age in ka, fraction of the plot area, vertical anchor). Placing them
# here rather than in a right-hand margin is what lets the data keep the full figure width.
LABEL_POS = {
    'LR04':         (132.5, 0.82, 'top'),
    'DSDP\n480/479': (5, 0.38, 'bottom'),
    'NH22P':        (16, 0.12, 'top'),
}

fig = plt.figure(figsize=(9.6, 6.4))
gs = fig.add_gridspec(2, height_ratios=[0.42, 6], hspace=0.02)

ribbon = fig.add_subplot(gs[0])
mis_ribbon(ribbon)

host = fig.add_subplot(gs[1])
host.set_xlim(XMIN, XMAX)
interglacial_bands(host)
host.set_yticks([])
host.spines[['left', 'right', 'top']].set_visible(False)
host.set_xlabel('AGE [ka]', size=10.5, labelpad=4)
host.xaxis.set_major_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 20)))
host.xaxis.set_minor_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 5)))
host.tick_params(axis='x', direction='out', labelsize=9.5)

# top -> bottom: (name, core or None for LR04, colour, ticks, axis side, spine offset in points)
traces = [
    ('LR04',         None,     LR04_C, LR04_TICKS,  'right', 0),
    ('DSDP\n480/479', d480_479, TEAL,   DD_TICKS,    'left',  0),
    ('NH22P',        nh22p,    GREEN,  DD_TICKS_NH, 'left',  50),
]

for (name, core, col, ticks, side, offset), slot in zip(traces, SLOTS):
    ax = host.twinx()
    ax.set_xlim(XMIN, XMAX)

    # Every trace is clipped to <= XMAX. The LR04 stack runs to 5320 ka, and DSDP-480/479's own
    # record reaches 152.47 ka -- without the clip both draw past the end of the age axis.
    if core is None:
        m = lr04_age <= XMAX
        ax.set_ylim(slot_ylim(2.95, 5.15, *slot, invert=True))
        ax.plot(lr04_age[m], lr04_d18O[m], color=col, lw=1.2, zorder=3)
        unit = u'$\\delta^{18}$O$_b$ [‰]'
    else:
        c = core.sel(age=slice(None, XMAX))
        ax.set_ylim(slot_ylim(*DD_LIM, *slot))
        ax.fill_between(c.age, c.dDraw - c.stdev, c.dDraw + c.stdev,
                        color=tint(col, 0.55), lw=0, alpha=0.9, zorder=3)
        ax.plot(c.age, c.dDraw, color=col, lw=1.4, zorder=4, solid_capstyle='round')
        unit = u'$\\delta$D$_{C30}$ [‰]'

    # only this trace's own axis is drawn, in this trace's colour, spanning only its own range
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_ticks_position(side)
    ax.yaxis.set_label_position(side)
    ax.spines[side].set_position(('outward', offset))
    ax.spines[side].set_bounds(min(ticks), max(ticks))
    ax.spines[side].set_color(col)
    for s in ('top', 'bottom', 'left', 'right'):
        if s != side:
            ax.spines[s].set_visible(False)
    ax.tick_params(axis='y', colors=col, labelsize=9, direction='out')
    ax.tick_params(axis='x', bottom=False, labelbottom=False)
    # pin the unit label to THIS trace's bracket; left at the default it centres on the whole
    # plot area and all three pile up in the middle, tied to nothing
    ax.set_ylabel(unit, size=9, color=col,
                  rotation=90 if side == 'left' else 270,
                  labelpad=4 if side == 'left' else 14,
                  y=(slot[0] + slot[1]) / 2, va='center')

    lx, ly, lva = LABEL_POS[name]
    blend = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(lx, ly, name, transform=blend, ha='left', va=lva, size=10, weight='bold',
            color=col, zorder=9)

ribbon.set_xlim(XMIN, XMAX)
interglacial_bands(host, labels=True, label_y=0.008)

fig.savefig(f'{opath}/dDraw_timeseries_stacked.pdf', bbox_inches='tight')

## dDp timeseries

In [ ]:
# --- dDp axis limits -------------------------------------------------------------------------
# Percentile bands are computed with np.nanpercentile directly in the plotting cell below.
# The previous version hardcoded `iters = 1000` and indexed into a sorted ensemble, which is
# only correct while both dDp sheets happen to be exactly 1000 members wide — that assumption
# silently broke once before, when the NH22P sheet was 1020 wide and the 97.5th-percentile band
# was drawn too narrow. See DATA_MANIFEST.md section 1b.

# Widened from the old (-75,-38) / (-73,-38): those clipped the 2-sigma envelope, which reaches
# -77.60 and -36.80 on DSDP-480/479 (5 points below, 2 above) and -35.63 on NH22P (2 above).
# A clipped uncertainty band reads as a narrower band, which is the one thing it must not do.
DDP_LIM   = (-78, -35)
DDP_TICKS = [-75,-65,-55,-45,-35]   # interior only, same rule as the dDraw figure

# NH22P's own bracket in the stacked version, offset from DSDP's by the same +5 used for
# DD_TICKS_NH -- keeps the two dDp/dDraw stacked figures visually consistent siblings.
DDP_TICKS_NH = [-75,-65,-55,-45,-35]

In [ ]:
# === dDp timeseries ==========================================================================
# Same structural fixes as the dDraw figure -- MIS ribbon in its own gridspec row, interior
# ticks on both axes, widened ylim -- AND now the same palette: each core takes its own TEAL /
# GREEN line colour instead of the shared gold, with the 1-sigma/2-sigma bands as two tints of
# that colour (tint() from the shared cell) rather than a fixed sig1/sig2 pair. The panel name
# is set in the core's colour too, matching Version A -- identity is carried the same way in
# both figures now.

fig = plt.figure(figsize=(9, 6))
gs = fig.add_gridspec(3, height_ratios=[0.5, 5, 5], hspace=0)

ribbon = fig.add_subplot(gs[0])
mis_ribbon(ribbon)

axes = []
for i, (name, core, col) in enumerate([('DSDP-480/479', d480_479, TEAL),
                                       ('NH22P', nh22p, GREEN)]):
    ax = fig.add_subplot(gs[i + 1], sharex=axes[0] if axes else None)
    axes.append(ax)

    interglacial_bands(ax)

    # benthic stack, behind the proxy on a twin axis
    axr = ax.twinx()
    axr.plot(lr04_age, lr04_d18O, color=LR04_C, lw=1.1, zorder=2)
    axr.set(xlim=(XMIN, XMAX), ylim=LR04_LIM, yticks=LR04_TICKS)
    axr.set_ylabel(u'LR04 $\\delta^{18}$O$_b$ [‰]', rotation=270, labelpad=16, size=10,
                   color=LR04_C)
    axr.tick_params(axis='y', colors=LR04_C, labelsize=9, direction='out')
    axr.spines['right'].set_color(LR04_C)
    axr.spines[['top', 'left', 'bottom']].set_visible(False)
    ax.set_zorder(axr.get_zorder() + 1)
    ax.patch.set_visible(False)

    # bands as percentiles over the ensemble axis -- correct at any ensemble width. Both bands
    # are tints of the core's own line colour: 2-sigma lighter (closer to white) than 1-sigma,
    # same relationship sig1/sig2 had, but tied to the core instead of a fixed gold pair.
    p = np.nanpercentile(core['dDp'], [2.5, 16, 84, 97.5], axis=1)   # -> (4, n_age)
    ax.fill_between(core.age, p[0], p[3], color=tint(col, 0.70), lw=0, alpha=0.9,
                    label='2$\\sigma$', zorder=2)
    ax.fill_between(core.age, p[1], p[2], color=tint(col, 0.40), lw=0, alpha=0.9,
                    label='1$\\sigma$', zorder=3)
    # the central line is the MEDIAN of the ensemble, not the mean -- see CLAUDE.md
    ax.plot(core.age, np.nanmedian(core['dDp'], axis=1), color=col, lw=1.5, zorder=4,
            label='median')

    ax.set(xlim=(XMIN, XMAX), ylim=DDP_LIM, yticks=DDP_TICKS)
    ax.set_ylabel(u'$\\delta$D$_{prec}$ [‰]', size=10.5, labelpad=4)
    ax.tick_params(axis='both', direction='out', labelsize=9.5)
    ax.yaxis.set_minor_locator(MultipleLocator(5))
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_position(('outward', 2))
    # panel name in the core's own colour, so identity is never colour-alone
    ax.text(0.012, 0.93, name, transform=ax.transAxes, ha='left', va='top', size=12,
            weight='bold', color=col, zorder=7)

axes[0].tick_params(labelbottom=False)
bottom = axes[-1]
bottom.set_xlabel('AGE [ka]', size=10.5, labelpad=4)
bottom.xaxis.set_major_locator(MultipleLocator(20))
ribbon.set_xlim(XMIN, XMAX)

interglacial_bands(bottom, labels=True)

fig.savefig(f'{opath}/dDp_timeseries.pdf', bbox_inches='tight')

In [ ]:
# === VERSION B: dDp, stacked overlapping traces, one plot area ===============================
# Same construction as the dDraw stacked figure (LR04 first, equal-height slots, labels inside
# the plot area, everything clipped to <= XMAX) applied to dDp instead of dDraw.
# The core traces are median + 1-sigma/2-sigma tinted bands

SLOTS = [(0.56, 1.00), (0.35, 0.72), (0.00, 0.44)]   # identical to the dDraw stacked figure

LABEL_POS = {
    'LR04':         (132.5, 0.82, 'top'),
    'DSDP\n480/479': (-7, 0.38, 'bottom'),
    'NH22P':        (16, 0.12, 'top'),
}

fig = plt.figure(figsize=(9.6, 6.4))
gs = fig.add_gridspec(2, height_ratios=[0.42, 6], hspace=0.02)

ribbon = fig.add_subplot(gs[0])
mis_ribbon(ribbon)

host = fig.add_subplot(gs[1])
host.set_xlim(XMIN, XMAX)
interglacial_bands(host)
host.set_yticks([])
host.spines[['left', 'right', 'top']].set_visible(False)
host.set_xlabel('AGE [ka]', size=10.5, labelpad=4)
host.xaxis.set_major_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 20)))
host.xaxis.set_minor_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 5)))
host.tick_params(axis='x', direction='out', labelsize=9.5)

traces = [
    ('LR04',               None,     LR04_C, LR04_TICKS,   'left', 0),
    ('DSDP\n480/479',      d480_479, TEAL,   DDP_TICKS,    'right',  40),
    ('NH22P',              nh22p,    GREEN,  DDP_TICKS_NH, 'right',  0),
]

for (name, core, col, ticks, side, offset), slot in zip(traces, SLOTS):
    ax = host.twinx()
    ax.set_xlim(XMIN, XMAX)

    if core is None:
        m = lr04_age <= XMAX
        ax.set_ylim(slot_ylim(2.95, 5.15, *slot, invert=True))
        ax.plot(lr04_age[m], lr04_d18O[m], color=col, lw=1.2, zorder=3)
        unit = u'LR04 $\\delta^{18}$O$_b$ [‰]'
    else:
        c = core.sel(age=slice(None, XMAX))
        p = np.nanpercentile(c['dDp'], [2.5, 16, 84, 97.5], axis=1)   # -> (4, n_age)
        ax.set_ylim(slot_ylim(*DDP_LIM, *slot))
        ax.fill_between(c.age, p[0], p[3], color=tint(col, 0.70), lw=0, alpha=0.9, zorder=2)
        ax.fill_between(c.age, p[1], p[2], color=tint(col, 0.40), lw=0, alpha=0.9, zorder=3)
        ax.plot(c.age, np.nanmedian(c['dDp'], axis=1), color=col, lw=1.4, zorder=4,
                solid_capstyle='round')
        unit = u'$\\delta$D$_{prec}$ [‰]'

    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_ticks_position(side)
    ax.yaxis.set_label_position(side)
    ax.spines[side].set_position(('outward', offset))
    ax.spines[side].set_bounds(min(ticks), max(ticks))
    ax.spines[side].set_color(col)
    for s in ('top', 'bottom', 'left', 'right'):
        if s != side:
            ax.spines[s].set_visible(False)
    ax.tick_params(axis='y', colors=col, labelsize=9, direction='out')
    ax.tick_params(axis='x', bottom=False, labelbottom=False)
    ax.set_ylabel(unit, size=9, color=col,
                  rotation=90 if side == 'left' else 270,
                  labelpad=4 if side == 'left' else 14,
                  y=(slot[0] + slot[1]) / 2, va='center')

    lx, ly, lva = LABEL_POS[name]
    blend = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(lx, ly, name, transform=blend, ha='left', va=lva, size=14, weight='bold',
            color=col, zorder=9)

ribbon.set_xlim(XMIN, XMAX)
interglacial_bands(host, labels=True, label_y=0.008)

fig.savefig(f'{opath}/dDp_timeseries_stacked.pdf', bbox_inches='tight')

## δDp vs. LR04 δ¹⁸O — significance

Ports the LR04 half of `scripts/matlab/SST_dD_correlation.m` (May 2022), which was written once
and never wired into a notebook. Its SST-vs-δDp half (`nh22p_uk_sst.mat`) is dropped here: that
file doesn't exist anywhere in this repo, `data/external/`, or `~/OneDrive`, and its provenance
was never documented — see `CLAUDE.md`.

In [ ]:
# --- Ebisuzaki (1997) phase-randomization significance test ---------------------------------
#
# corrcoef's p-value assumes independent samples. Both dDp and LR04 d18O are autocorrelated
# time series, so a naive p-value overstates significance -- this is why the MATLAB pipeline
# never uses corrcoef for series like these (see CLAUDE.md). Ebisuzaki (1997, J. Climate 10,
# 2147-2153) tests against a null built from `nsim` surrogate pairs: both x and y are rebuilt
# from their own power spectra with randomized phases (so each surrogate keeps its parent's
# autocorrelation structure but not any real coupling between them), and F is the fraction of
# surrogate |r| that exceed the real |r|.
#
# This is a from-scratch NumPy port of the published algorithm, not a port of the vendored
# ebisuzaki.m (~/Documents/MATLAB/toolbox/ebisuzaki.m, not distributed with this repo -- see
# CLAUDE.md's note on vendoring). It reproduces the same phase-randomization construction
# (including MATLAB's even/odd-length branch, needed to keep the inverse FFT real), but it will
# NOT bit-match the old script's printed numbers: MATLAB's RNG and NumPy's default_rng are
# different algorithms, so a fixed seed only buys run-to-run reproducibility here, not
# cross-language reproducibility.

def ebisuzaki_test(x, y, sig=0.05, nsim=10000, seed=22459):
    """Ebisuzaki (1997) phase-randomization test for correlation between two autocorrelated series.

    Returns (r_obs, F, critical_r): the observed Pearson r, the fraction of surrogate |r| that
    exceed |r_obs| (compare this to `sig` -- it IS the significance, not a p-value needing
    further correction), and the critical |r| at the `sig` level.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape:
        raise ValueError('x and y must be the same shape')
    n = x.size
    n2 = n // 2

    r_obs = np.corrcoef(x, y)[0, 1]
    modx = np.abs(np.fft.fft(x))
    mody = np.abs(np.fft.fft(y))
    rng = np.random.default_rng(seed)

    def _phase_randomized(mod):
        if n % 2 == 0:
            phases = rng.uniform(-np.pi, np.pi, n2 - 1)
            angles = np.concatenate(([0.0], phases, [0.0], -phases[::-1]))
        else:
            phases = rng.uniform(-np.pi, np.pi, n2)
            angles = np.concatenate(([0.0], phases, -phases[::-1]))
        s = np.real(np.fft.ifft(mod * np.exp(1j * angles)))
        return (s - s.mean()) / s.std()

    r_sim = np.empty(nsim)
    for i in range(nsim):
        r_sim[i] = np.corrcoef(_phase_randomized(modx), _phase_randomized(mody))[0, 1]

    F = np.mean(np.abs(r_sim) > np.abs(r_obs))
    critical_r = np.sort(np.abs(r_sim))[int(np.floor(nsim * (1 - sig))) - 1]
    return r_obs, F, critical_r

In [ ]:
# --- Shared LR04 correlation-scatter drawing --------------------------------------------------
# Used by both standalone core panels below AND the composite figure further down, so the
# trendline/legend/title/axis-side styling can only be defined once and can't drift between the
# three uses. The keyword args' defaults reproduce the standalone panels' original look; the
# composite figure overrides them (see that cell) for the narrower, paired-panel layout it needs.

def plot_lr04_scatter(ax, points, lr04_full, dDp_full, trend_r, title, title_color,
                       trend_label=None, title_loc='above', y_side='left',
                       show_xlabel=True, trend_in_legend=True):
    """Draw a δDp vs. LR04 δ18O scatter with a full-record trendline.

    points: list of (label, x, y, facecolor) tuples, one per age-window scatter already
    computed by the caller.
    lr04_full / dDp_full: this core's FULL record (every sample, not just the age windows in
    `points`) -- the trendline is a linear fit to all of it, independent of how it's windowed
    for the significance tests.
    trend_r: the full-record correlation coefficient, already computed by the caller via
    ebisuzaki_test -- correlation is invariant to standardization, so its r_obs IS the plain
    Pearson r, no need to recompute it here. Shown either in the legend (trend_in_legend=True,
    via `trend_label`, which must then be given) or written directly beside the trendline itself
    (trend_in_legend=False).
    title_loc: 'above' (default) draws the title above the legend, above the axes -- the
    standalone panels' layout. 'inside' draws it in the axes' own top-left corner instead, for
    panels too narrow to spare the vertical room 'above' costs.
    y_side: 'left' (default) or 'right' -- which spine carries the y ticks and axis label.
    show_xlabel: False hides the x tick labels and axis label entirely, for a panel sharing its
    x-axis with a sibling panel below it.
    """
    handles, labels, colors = [], [], []
    for label, x, y, fc in points:
        h = ax.scatter(x, y, s=55, fc=fc, ec='k', lw=0.6, zorder=3)
        handles.append(h)
        labels.append(label)
        colors.append(fc)

    slope, intercept = np.polyfit(lr04_full, dDp_full, 1)
    xline = np.array([np.min(lr04_full), np.max(lr04_full)])
    yline = slope * xline + intercept
    trend, = ax.plot(xline, yline, color='0.25', ls='--', lw=1.3, zorder=4)

    # both the inside-axes title and the inline trend annotation sit ON TOP of the scatter
    # cloud, unlike their 'above the axes'/legend counterparts -- a light halo keeps them
    # readable over whatever point happens to land underneath
    halo = dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5)

    if trend_in_legend:
        handles.append(trend)
        labels.append(trend_label)
        colors.append('0.25')
    else:
        # r written directly beside the line's own end rather than living in the legend -- keeps
        # the legend down to just the tested windows when a panel has no room to spare a third
        # entry (or third row, at ncol=2)
        ax.annotate(f'r={trend_r:.2f}', xy=(xline[-1], yline[-1]), xytext=(-4, 5),
                    textcoords='offset points', ha='right', va='bottom', size=8, color='0.25',
                    zorder=5, bbox=halo)

    if show_xlabel:
        ax.set_xlabel(u'LR04 $\\delta^{18}$O$_b$ [‰]', size=9.5)
    else:
        ax.tick_params(labelbottom=False)
    ax.set_ylabel(u'$\\delta$D$_{prec}$ [‰]', size=9.5)
    ax.tick_params(labelsize=8.5)
    if y_side == 'right':
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')

    if title_loc == 'inside':
        ax.text(0.04, 0.95, title, transform=ax.transAxes, ha='left', va='top', size=11,
                weight='bold', color=title_color, zorder=7, bbox=halo)
    else:
        # Title above the legend, legend above the axes -- both in axes-fraction coords so they
        # stack in a fixed relationship regardless of the panel's own size.
        ax.text(0.5, 1.34, title, transform=ax.transAxes, ha='center', va='bottom', size=11,
                weight='bold', color=title_color)
    leg = ax.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2,
                    fontsize=7.5, frameon=False, handletextpad=0.4, columnspacing=1.0)
    # legend text colour matches its own marker/line colour, not the default black
    for text, c in zip(leg.get_texts(), colors):
        text.set_color(c)
    return leg

In [ ]:
# --- DSDP-480/479: dDp vs. LR04 d18O ------------------------------------------------------------
# Reproduces SST_dD_correlation.m's DSDP block: all three windows were tested there (full record
# plus the two age-split halves) -- unlike NH22P's, none of these are commented out. Same
# median-not-mean and interp1-extrapolate notes as the NH22P cell below apply (lr04_interp is
# defined once, in the LOAD PROXY DATA cell). The full-record window also supplies the linear
# trendline drawn on top of the two age-split scatters.

d480_479_lr04 = xr.DataArray(lr04_interp(d480_479.age.values), coords={'age': d480_479.age},
                              dims='age')
d480_479_dDp_med = d480_479.dDp.median(dim='ensemble_n_dDp')

DSDP_WINDOWS = {
    '0-35 ka':         (0, 35),   # present to LGM
    '35-152 ka':       (35, 152), # MIS 3 to LIG
    'full (0-152 ka)': (0, 152),  # full record
}

dsdp_results = {}
for label, (lo, hi) in DSDP_WINDOWS.items():
    sub_lr04 = d480_479_lr04.sel(age=slice(lo, hi))
    sub_dDp = d480_479_dDp_med.sel(age=slice(lo, hi))
    r_obs, F, r_crit = ebisuzaki_test(sub_lr04.values, sub_dDp.values)
    dsdp_results[label] = (r_obs, F, r_crit)
    print(f'DSDP-480/479 dDp vs. LR04 d18O R ({lo}-{hi} ka): '
          f'r={r_obs:.3f}, F={F:.3f}, critical|r|={r_crit:.3f}')

dsdp_young_lr04 = d480_479_lr04.sel(age=slice(*DSDP_WINDOWS['0-35 ka']))
dsdp_young_dDp = d480_479_dDp_med.sel(age=slice(*DSDP_WINDOWS['0-35 ka']))
dsdp_old_lr04 = d480_479_lr04.sel(age=slice(*DSDP_WINDOWS['35-152 ka']))
dsdp_old_dDp = d480_479_dDp_med.sel(age=slice(*DSDP_WINDOWS['35-152 ka']))
dsdp_full_lr04 = d480_479_lr04.sel(age=slice(*DSDP_WINDOWS['full (0-152 ka)']))
dsdp_full_dDp = d480_479_dDp_med.sel(age=slice(*DSDP_WINDOWS['full (0-152 ka)']))
r_y, F_y, _ = dsdp_results['0-35 ka']
r_o, F_o, _ = dsdp_results['35-152 ka']
r_f, F_f, _ = dsdp_results['full (0-152 ka)']

dsdp_points = [
    (f'35-152 ka: r={r_o:.2f}, F={F_o:.3f}', dsdp_old_lr04.values, dsdp_old_dDp.values, TEAL),
    (f'0-35 ka: r={r_y:.2f}, F={F_y:.3f}', dsdp_young_lr04.values, dsdp_young_dDp.values,
     tint(TEAL, 0.55)),
]
dsdp_trend_label = f'full record: r={r_f:.2f}, F={F_f:.3f}'

fig, ax = plt.subplots(figsize=(4.6, 4.7))
plot_lr04_scatter(ax, dsdp_points, dsdp_full_lr04.values, dsdp_full_dDp.values, r_f,
                  'DSDP-480/479', TEAL, trend_label=dsdp_trend_label)
fig.savefig(f'{opath}/dsdp480_479_dDp_lr04_correlation.pdf', bbox_inches='tight')

In [ ]:
# --- NH22P: dDp vs. LR04 d18O -----------------------------------------------------------------
# Reproduces SST_dD_correlation.m's NH22P block; extended here to also test the full record
# (the 2022 script never did -- see the DSDP cell above for the analogous full-record test it
# did run). Same median-not-mean and interp1-extrapolate notes as the DSDP cell above apply
# (lr04_interp is defined once, in the LOAD PROXY DATA cell). The full-record window also
# supplies the linear trendline drawn on top of the two age-split scatters.

nh22p_lr04 = xr.DataArray(lr04_interp(nh22p.age.values), coords={'age': nh22p.age}, dims='age')
nh22p_dDp_med = nh22p.dDp.median(dim='ensemble_n_dDp')

NH22P_WINDOWS = {
    '0-35 ka':         (0, 35),   # present to LGM
    '35-145 ka':       (35, 145), # MIS 3 to LIG
    'full (0-145 ka)': (0, 145),  # full record
}

nh22p_results = {}
for label, (lo, hi) in NH22P_WINDOWS.items():
    sub_lr04 = nh22p_lr04.sel(age=slice(lo, hi))
    sub_dDp = nh22p_dDp_med.sel(age=slice(lo, hi))
    r_obs, F, r_crit = ebisuzaki_test(sub_lr04.values, sub_dDp.values)
    nh22p_results[label] = (r_obs, F, r_crit)
    print(f'NH22P dDp vs. LR04 d18O R ({lo}-{hi} ka): '
          f'r={r_obs:.3f}, F={F:.3f}, critical|r|={r_crit:.3f}')

nh22p_young_lr04 = nh22p_lr04.sel(age=slice(*NH22P_WINDOWS['0-35 ka']))
nh22p_young_dDp = nh22p_dDp_med.sel(age=slice(*NH22P_WINDOWS['0-35 ka']))
nh22p_old_lr04 = nh22p_lr04.sel(age=slice(*NH22P_WINDOWS['35-145 ka']))
nh22p_old_dDp = nh22p_dDp_med.sel(age=slice(*NH22P_WINDOWS['35-145 ka']))
nh22p_full_lr04 = nh22p_lr04.sel(age=slice(*NH22P_WINDOWS['full (0-145 ka)']))
nh22p_full_dDp = nh22p_dDp_med.sel(age=slice(*NH22P_WINDOWS['full (0-145 ka)']))
r_y, F_y, _ = nh22p_results['0-35 ka']
r_o, F_o, _ = nh22p_results['35-145 ka']
r_f, F_f, _ = nh22p_results['full (0-145 ka)']

nh22p_points = [
    (f'35-145 ka: r={r_o:.2f}, F={F_o:.3f}', nh22p_old_lr04.values, nh22p_old_dDp.values, GREEN),
    (f'0-35 ka: r={r_y:.2f}, F={F_y:.3f}', nh22p_young_lr04.values, nh22p_young_dDp.values,
     tint(GREEN, 0.55)),
]
nh22p_trend_label = f'full record: r={r_f:.2f}, F={F_f:.3f}'

fig, ax = plt.subplots(figsize=(4.6, 4.7))
plot_lr04_scatter(ax, nh22p_points, nh22p_full_lr04.values, nh22p_full_dDp.values, r_f,
                  'NH22P', GREEN, trend_label=nh22p_trend_label)
fig.savefig(f'{opath}/nh22p_dDp_lr04_correlation.pdf', bbox_inches='tight')

In [ ]:
# === COMPOSITE: dDp timeseries (Version B) + LR04 correlation scatter column =================
# Same Version B construction as the dDp timeseries cell above, with the two LR04-correlation
# scatter panels added as their own stacked column on the right, so the whole figure reads as
# one rectangle: the timeseries takes the full height on the left, the two roughly-square
# scatter panels stack to match it on the right. They are NOT row-aligned with the timeseries
# traces underneath them -- Version B has no per-core row structure to align to (that's what
# Version A's two-panel layout is for), so the right column is simply sized to match the left
# column's total height instead.

fig = plt.figure(figsize=(13.8, 6.4))
outer = fig.add_gridspec(1, 2, width_ratios=[6.6, 3.6], wspace=0.34)

# --- left: dDp Version B timeseries, identical construction to the cell above ----------------
gsL = outer[0, 0].subgridspec(2, 1, height_ratios=[0.42, 6], hspace=0.02)

ribbon = fig.add_subplot(gsL[0])
mis_ribbon(ribbon)

host = fig.add_subplot(gsL[1])
host.set_xlim(XMIN, XMAX)
interglacial_bands(host)
host.set_yticks([])
host.spines[['left', 'right', 'top']].set_visible(False)
host.set_xlabel('AGE [ka]', size=10.5, labelpad=4)
host.xaxis.set_major_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 20)))
host.xaxis.set_minor_locator(FixedLocator(np.arange(XMIN, XMAX + 1, 5)))
host.tick_params(axis='x', direction='out', labelsize=9.5)

SLOTS = [(0.56, 1.00), (0.35, 0.72), (0.00, 0.44)]
LABEL_POS = {
    'LR04':         (132.5, 0.82, 'top'),
    'DSDP\n480/479': (-7, 0.38, 'bottom'),
    'NH22P':        (16, 0.12, 'top'),
}
traces = [
    ('LR04',          None,     LR04_C, LR04_TICKS,   'left',  0),
    ('DSDP\n480/479', d480_479, TEAL,   DDP_TICKS,    'right', 40),
    ('NH22P',         nh22p,    GREEN,  DDP_TICKS_NH, 'right', 0),
]

for (name, core, col, ticks, side, offset), slot in zip(traces, SLOTS):
    ax = host.twinx()
    ax.set_xlim(XMIN, XMAX)
    if core is None:
        m = lr04_age <= XMAX
        ax.set_ylim(slot_ylim(2.95, 5.15, *slot, invert=True))
        ax.plot(lr04_age[m], lr04_d18O[m], color=col, lw=1.2, zorder=3)
        unit = u'LR04 $\\delta^{18}$O$_b$ [‰]'
    else:
        c = core.sel(age=slice(None, XMAX))
        p = np.nanpercentile(c['dDp'], [2.5, 16, 84, 97.5], axis=1)
        ax.set_ylim(slot_ylim(*DDP_LIM, *slot))
        ax.fill_between(c.age, p[0], p[3], color=tint(col, 0.70), lw=0, alpha=0.9, zorder=2)
        ax.fill_between(c.age, p[1], p[2], color=tint(col, 0.40), lw=0, alpha=0.9, zorder=3)
        ax.plot(c.age, np.nanmedian(c['dDp'], axis=1), color=col, lw=1.4, zorder=4,
                solid_capstyle='round')
        unit = u'$\\delta$D$_{prec}$ [‰]'

    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_ticks_position(side)
    ax.yaxis.set_label_position(side)
    ax.spines[side].set_position(('outward', offset))
    ax.spines[side].set_bounds(min(ticks), max(ticks))
    ax.spines[side].set_color(col)
    for s in ('top', 'bottom', 'left', 'right'):
        if s != side:
            ax.spines[s].set_visible(False)
    ax.tick_params(axis='y', colors=col, labelsize=9, direction='out')
    ax.tick_params(axis='x', bottom=False, labelbottom=False)
    ax.set_ylabel(unit, size=9, color=col,
                  rotation=90 if side == 'left' else 270,
                  labelpad=4 if side == 'left' else 14,
                  y=(slot[0] + slot[1]) / 2, va='center')

    lx, ly, lva = LABEL_POS[name]
    blend = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(lx, ly, name, transform=blend, ha='left', va=lva, size=14, weight='bold',
            color=col, zorder=9)

ribbon.set_xlim(XMIN, XMAX)
interglacial_bands(host, labels=True, label_y=0.008)

# --- right: LR04 correlation scatter panels, stacked, sharing one x-axis ---------------------
# Title moves inside each axes (top-left) and the full-record trend is annotated on the line
# itself rather than living in the legend -- both free the vertical room 'above the axes' would
# otherwise need, which this narrower column doesn't have to spare. The y-axis moves to each
# panel's own right spine, matching the outward-facing-axis convention the timeseries panels
# already use (their own axis brackets sit at the LEFT/RIGHT outer edges of that column, not in
# the middle) -- so the gap between the two columns stays clear of tick labels on both sides.
# Sharing the x-axis means only the bottom panel (NH22P) needs its own tick labels and title.
gsR = outer[0, 1].subgridspec(2, 1, hspace=0.2)

axD = fig.add_subplot(gsR[0])
dsdp_r_full = dsdp_results['full (0-152 ka)'][0]
plot_lr04_scatter(axD, dsdp_points, dsdp_full_lr04.values, dsdp_full_dDp.values, dsdp_r_full,
                  'DSDP-480/479', TEAL, title_loc='inside', y_side='right',
                  show_xlabel=False, trend_in_legend=False)

axN = fig.add_subplot(gsR[1], sharex=axD)
nh22p_r_full = nh22p_results['full (0-145 ka)'][0]
plot_lr04_scatter(axN, nh22p_points, nh22p_full_lr04.values, nh22p_full_dDp.values, nh22p_r_full,
                  'NH22P', GREEN, title_loc='inside', y_side='right', trend_in_legend=False)

fig.savefig(f'{opath}/dDp_timeseries_with_lr04_correlation.pdf', bbox_inches='tight')